# Resume Protein ProGen Pretraining

This notebook resumes a checkpoint produced by `03_pretrain_protein_from_scratch.ipynb`. It restores tokenizer state, model config, optimizer state, and corpus metadata, then continues training with the same repo pretrain helpers.

It supports:
- single GPU or CPU by default;
- automatic DataParallel when multiple local GPUs are visible in the notebook session;
- DDP when the kernel is launched under torchrun and WORLD_SIZE, RANK, and LOCAL_RANK are set.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    LEGACY_PROTEIN_MODEL_FAMILY,
    MDCDecoderModel,
    MDCModelConfig,
    compute_mdc_causal_lm_loss,
    count_trainable_parameters,
    create_protein_lm_dataloader,
    create_streaming_protein_lm_dataloader,
    discover_protein_train_text_paths,
    evaluate_mdc_causal_lm_batch_loss,
    extract_protein_backbone_config,
    generate_protein_text,
    is_supported_protein_checkpoint_family,
    load_protein_corpus_text_parts,
    load_protein_pretrain_checkpoint,
    prepare_mdc_training_runtime,
    save_protein_pretrain_checkpoint,
    set_mdc_data_loader_epoch,
    split_protein_corpus_text,
)
from libs.core.pretrain.training_config import (
    apply_protein_training_optimizer_settings,
    build_protein_training_data_config,
    create_protein_training_optimizer,
    describe_protein_training_optimizers,
    load_protein_training_config,
)
from libs.data.training.tokenizer import SequenceTokenizer

PROJECT_ROOT

In [ ]:
TRAINING_CONFIG = load_protein_training_config(PROJECT_ROOT)
PATHS_CONFIG = TRAINING_CONFIG["paths"]
DATA_CONFIG = TRAINING_CONFIG["data"]
TRAINING_SETTINGS = TRAINING_CONFIG["training"]
OPTIMIZER_CONFIG = TRAINING_CONFIG["optimizer"]
RUNTIME_CONFIG = TRAINING_CONFIG["runtime"]
RESUME_CONFIG = TRAINING_CONFIG["resume"]
MINIO_CONFIG = TRAINING_CONFIG["minio"]
MINIO_DATA_CONFIG = build_protein_training_data_config(PROJECT_ROOT, TRAINING_CONFIG)

CHECKPOINT_DIR = PATHS_CONFIG["checkpoint_dir"]
RESUME_CHECKPOINT_PATH = RESUME_CONFIG["checkpoint_path"]
OUTPUT_CHECKPOINT_PATH = RESUME_CONFIG["output_checkpoint_path"]
BEST_CHECKPOINT_PATH = RESUME_CONFIG["best_checkpoint_path"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TEXT_PATH = PATHS_CONFIG["train_text_path"]
TRAIN_PART_GLOB = DATA_CONFIG["train_part_glob"]
PREFER_LOCAL_TRAIN_PARTS = DATA_CONFIG["prefer_local_train_parts"]
STREAM_LOCAL_TRAIN_PARTS = DATA_CONFIG["stream_local_train_parts"]
MINIO_TRAIN_PARTS_PREFIX_URI = MINIO_CONFIG["train_parts_prefix_uri"]
MINIO_TRAIN_PART_URIS = MINIO_CONFIG["train_part_uris"]
TRAIN_PART_CACHE_DIR = PATHS_CONFIG["train_part_cache_dir"]
KEEP_DOWNLOADED_TRAIN_PARTS = DATA_CONFIG["keep_downloaded_train_parts"]

NUM_EPOCHS = TRAINING_SETTINGS["num_epochs"]
TRAIN_RATIO = DATA_CONFIG["train_ratio"]
BATCH_SIZE = DATA_CONFIG["batch_size"]
STRIDE = TRAINING_CONFIG["model"]["stride"]
LEARNING_RATE = OPTIMIZER_CONFIG["learning_rate"]
MUON_LEARNING_RATE = OPTIMIZER_CONFIG["muon_learning_rate"]
WEIGHT_DECAY = OPTIMIZER_CONFIG["weight_decay"]
OPTIMIZER_TYPE = OPTIMIZER_CONFIG["type"]
GRAD_CLIP_NORM = TRAINING_SETTINGS["grad_clip_norm"]
EVAL_FREQ = TRAINING_SETTINGS["eval_freq"]
EVAL_BATCHES = TRAINING_SETTINGS["eval_batches"]
NUM_WORKERS = DATA_CONFIG["num_workers"]
PIN_MEMORY = DATA_CONFIG["pin_memory"]
REQUESTED_DEVICE = RUNTIME_CONFIG["device"]
MULTI_GPU_MODE = RUNTIME_CONFIG["multi_gpu_mode"]
DDP_FIND_UNUSED_PARAMETERS = RUNTIME_CONFIG["ddp_find_unused_parameters"]
DATA_PARALLEL_DEVICE_IDS = RUNTIME_CONFIG["data_parallel_device_ids"]
RESTORE_OPTIMIZER_STATE = RESUME_CONFIG["restore_optimizer_state"]
OVERRIDE_OPTIMIZER_HYPERPARAMETERS = RESUME_CONFIG["override_optimizer_hyperparameters"]

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", os.environ.get("RANK", "0")))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
IS_DISTRIBUTED = WORLD_SIZE > 1
IS_MAIN_PROCESS = RANK == 0

if not RESUME_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing resume checkpoint: {RESUME_CHECKPOINT_PATH}")

checkpoint_meta = torch.load(RESUME_CHECKPOINT_PATH, map_location="cpu")
if not is_supported_protein_checkpoint_family(checkpoint_meta.get("model_family")):
    raise ValueError(
        "This resume notebook expects a protein pretrain checkpoint saved by the stage-2 flow. "
        f"Received: {checkpoint_meta.get('model_family')!r}"
    )

checkpoint_summary = {
    "train_config_path": str(TRAINING_CONFIG["config_path"]),
    "checkpoint_path": str(RESUME_CHECKPOINT_PATH),
    "checkpoint_family": checkpoint_meta.get("model_family") or LEGACY_PROTEIN_MODEL_FAMILY,
    "checkpoint_keys": sorted(checkpoint_meta.keys()),
    "optimizer_type": OPTIMIZER_TYPE,
    "restore_optimizer_state": RESTORE_OPTIMIZER_STATE,
    "override_optimizer_hyperparameters": OVERRIDE_OPTIMIZER_HYPERPARAMETERS,
    "rank": RANK,
    "local_rank": LOCAL_RANK,
    "world_size": WORLD_SIZE,
    "multi_gpu_mode": MULTI_GPU_MODE,
    "pin_memory": PIN_MEMORY,
    "num_workers": NUM_WORKERS,
}
checkpoint_summary

In [ ]:
tokenizer_map_path = Path(checkpoint_meta["tokenizer_map_path"])
if not tokenizer_map_path.exists():
    tokenizer_map_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer_map_path.write_text(
        json.dumps(checkpoint_meta["tokenizer_map"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
checkpoint_corpus_files = tuple(Path(path) for path in checkpoint_meta.get("corpus_files", ()) if Path(path).exists())
train_text_path = Path(checkpoint_meta.get("corpus_file") or TRAIN_TEXT_PATH)
if checkpoint_corpus_files:
    local_train_text_paths = checkpoint_corpus_files
elif train_text_path.exists() or any(train_text_path.parent.glob(TRAIN_PART_GLOB)):
    local_train_text_paths = discover_protein_train_text_paths(
        train_text_path,
        part_glob=TRAIN_PART_GLOB,
        prefer_parts=PREFER_LOCAL_TRAIN_PARTS,
    )
else:
    local_train_text_paths = ()

use_minio_train_parts = bool(MINIO_TRAIN_PARTS_PREFIX_URI or MINIO_TRAIN_PART_URIS)
use_streaming_loader = use_minio_train_parts or (STREAM_LOCAL_TRAIN_PARTS and len(local_train_text_paths) > 1)
corpus_text = ""
if not use_streaming_loader and local_train_text_paths:
    corpus_text = load_protein_corpus_text_parts(local_train_text_paths)
if not corpus_text and not use_streaming_loader:
    raise ValueError("A local train.txt or train_part_*.txt corpus is required unless MinIO train parts are configured.")
train_text, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO) if corpus_text else ("", "")

model_config_payload = dict(checkpoint_meta["model_config"])
if model_config_payload.get("layer_types") is not None:
    model_config_payload["layer_types"] = tuple(model_config_payload["layer_types"])
model_config = MDCModelConfig(**model_config_payload)
context_length = int(model_config.context_length)
stride = STRIDE or max(1, context_length // 2)

loader_kwargs = {
    "context_length": context_length,
    "stride": stride,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}
distributed_loader_kwargs = {
    "distributed": IS_DISTRIBUTED,
    "rank": RANK,
    "world_size": WORLD_SIZE,
}

if use_minio_train_parts:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer,
        prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
        part_uris=MINIO_TRAIN_PART_URIS or None,
        config=MINIO_DATA_CONFIG,
        cache_dir=TRAIN_PART_CACHE_DIR,
        keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
        shuffle_parts=True,
        shuffle_examples=True,
        seed=RANK,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_streaming_protein_lm_dataloader(
            tokenizer,
            prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
            part_uris=MINIO_TRAIN_PART_URIS or None,
            config=MINIO_DATA_CONFIG,
            cache_dir=TRAIN_PART_CACHE_DIR,
            keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
            shuffle_parts=False,
            shuffle_examples=False,
            seed=0,
            distributed=False,
            **loader_kwargs,
        )
        if IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "minio streaming"
elif use_streaming_loader:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer,
        part_paths=local_train_text_paths,
        shuffle_parts=True,
        shuffle_examples=True,
        seed=RANK,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_streaming_protein_lm_dataloader(
            tokenizer,
            part_paths=local_train_text_paths,
            shuffle_parts=False,
            shuffle_examples=False,
            seed=0,
            distributed=False,
            **loader_kwargs,
        )
        if IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "local part streaming"
else:
    train_loader = create_protein_lm_dataloader(
        train_text,
        tokenizer,
        shuffle=True,
        sampler_seed=0,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_protein_lm_dataloader(
            train_text,
            tokenizer,
            shuffle=False,
            distributed=False,
            **loader_kwargs,
        )
        if train_text and IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "in-memory split"

val_loader = (
    create_protein_lm_dataloader(
        val_text,
        tokenizer,
        shuffle=False,
        distributed=False,
        **loader_kwargs,
    )
    if val_text and IS_MAIN_PROCESS
    else None
)

def dataloader_size(loader):
    if loader is None:
        return "disabled"
    try:
        return len(loader)
    except TypeError:
        return "streaming"

if IS_MAIN_PROCESS:
    print(f"Corpus: {train_text_path}")
    print(f"Local corpus files: {len(local_train_text_paths)}")
    print(f"Train loader: {train_loader_description}")
    print(f"Tokenizer: {tokenizer_map_path} vocab={tokenizer.vocab_size}")
    print(f"Context length: {context_length} stride={stride}")
    print(f"Checkpoint family: {checkpoint_meta.get('model_family') or LEGACY_PROTEIN_MODEL_FAMILY}")
    print(f"Batches: train={dataloader_size(train_loader)} train_eval={dataloader_size(train_eval_loader)} val={dataloader_size(val_loader)}")
else:
    print(f"Rank {RANK}/{WORLD_SIZE} prepared distributed resume loader: {train_loader_description}")

In [ ]:
requested_device = REQUESTED_DEVICE
if requested_device == "auto":
    requested_device = "cuda" if torch.cuda.is_available() else "cpu"
runtime = prepare_mdc_training_runtime(
    MDCDecoderModel(model_config),
    device=requested_device,
    multi_gpu=MULTI_GPU_MODE,
    find_unused_parameters=DDP_FIND_UNUSED_PARAMETERS,
    data_parallel_device_ids=DATA_PARALLEL_DEVICE_IDS,
)
model = runtime.model
device = runtime.device

optimizer = create_protein_training_optimizer(
    model,
    OPTIMIZER_CONFIG,
    device=device,
)
optimizer_names = describe_protein_training_optimizers(optimizer)
try:
    checkpoint = load_protein_pretrain_checkpoint(
        RESUME_CHECKPOINT_PATH,
        model=model,
        optimizer=optimizer if RESTORE_OPTIMIZER_STATE else None,
        map_location=device,
    )
except ValueError as exc:
    if RESTORE_OPTIMIZER_STATE:
        raise ValueError(
            "Optimizer state in the checkpoint does not match train.yaml. "
            "Set resume.restore_optimizer_state=false to switch optimizer type or optimizer count."
        ) from exc
    raise

if OVERRIDE_OPTIMIZER_HYPERPARAMETERS:
    apply_protein_training_optimizer_settings(optimizer, OPTIMIZER_CONFIG)

start_epoch = int(checkpoint.get("epoch", 0))
global_step = int(checkpoint.get("global_step", 0))
tokens_seen = int(checkpoint.get("tokens_seen", 0))
train_losses = list(checkpoint.get("train_losses", []))
val_losses = list(checkpoint.get("val_losses", []))
best_val_loss = checkpoint.get("best_val_loss")
best_val_loss = math.inf if best_val_loss is None else float(best_val_loss)

runtime_summary = {
    "device": str(device),
    "distributed": runtime.distributed,
    "data_parallel": runtime.data_parallel,
    "rank": runtime.rank,
    "world_size": runtime.world_size,
    "optimizer_types": optimizer_names,
    "trainable_parameters": count_trainable_parameters(model),
}

if runtime.is_main_process:
    print(f"Resumed from epoch={start_epoch} step={global_step} tokens={tokens_seen:,} on {device}")
    print(f"Distributed: {runtime.distributed} data_parallel={runtime.data_parallel} world_size={runtime.world_size}")
    print(f"Optimizers: {optimizer_names}")
    print(f"Trainable parameters: {count_trainable_parameters(model):,}")

runtime_summary

In [ ]:
metrics_path = CHECKPOINT_DIR / "resume_training_metrics.jsonl"
step_eval_enabled = EVAL_FREQ > 0 and not runtime.distributed and train_eval_loader is not None
optimizer_list = list(optimizer) if isinstance(optimizer, (list, tuple)) else [optimizer]

def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )

def append_metrics(payload):
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def distributed_barrier():
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        torch.distributed.barrier()

def count_step_tokens(batch):
    token_count = torch.tensor(
        int(batch.attention_mask.sum().item()),
        device=device,
        dtype=torch.long,
    )
    if runtime.distributed:
        torch.distributed.all_reduce(token_count, op=torch.distributed.ReduceOp.SUM)
    return int(token_count.item())

def save_checkpoint(path, epoch, val_loss):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer,
        tokenizer_map_path=tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_val_loss) else best_val_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": context_length,
            "stride": stride,
            "learning_rate": LEARNING_RATE,
            "muon_learning_rate": MUON_LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer_type": OPTIMIZER_TYPE,
            "optimizer_types": optimizer_names,
            "resumed_from": str(RESUME_CHECKPOINT_PATH.resolve()),
            "restore_optimizer_state": RESTORE_OPTIMIZER_STATE,
            "override_optimizer_hyperparameters": OVERRIDE_OPTIMIZER_HYPERPARAMETERS,
            "multi_gpu_mode": MULTI_GPU_MODE,
            "num_workers": NUM_WORKERS,
            "pin_memory": PIN_MEMORY,
            "train_config_path": str(TRAINING_CONFIG["config_path"]),
        },
        extra={
            "corpus_file": str(train_text_path.resolve()),
            "corpus_files": [str(path.resolve()) for path in local_train_text_paths],
            "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
            "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
            "progen_config": extract_protein_backbone_config(checkpoint),
            "last_eval_val_loss": val_loss,
        },
    )

for epoch_offset in range(1, NUM_EPOCHS + 1):
    epoch = start_epoch + epoch_offset
    set_mdc_data_loader_epoch(train_loader, epoch - 1)
    model.train()
    for batch in train_loader:
        batch = move_batch(batch, device)
        for opt in optimizer_list:
            opt.zero_grad(set_to_none=True)
        logits = model(batch.input_ids, batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        for opt in optimizer_list:
            opt.step()

        global_step += 1
        tokens_seen += count_step_tokens(batch)

        if step_eval_enabled and global_step % EVAL_FREQ == 0:
            train_eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model,
                train_eval_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            val_loss = (
                evaluate_mdc_causal_lm_batch_loss(
                    model,
                    val_loader,
                    device=device,
                    max_batches=EVAL_BATCHES,
                )
                if val_loader is not None
                else float("nan")
            )
            train_losses.append(train_eval_loss)
            val_losses.append(val_loss)
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": train_eval_loss,
                "val_loss": val_loss,
            })
            if val_loader is not None and val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(BEST_CHECKPOINT_PATH, epoch, val_loss)
            print(f"epoch={epoch} step={global_step} train={train_eval_loss:.4f} val={val_loss:.4f}")

    distributed_barrier()

    if runtime.is_main_process:
        train_eval_loss = (
            evaluate_mdc_causal_lm_batch_loss(
                model,
                train_eval_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            if train_eval_loader is not None
            else float("nan")
        )
        val_loss = (
            evaluate_mdc_causal_lm_batch_loss(
                model,
                val_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            if val_loader is not None
            else float("nan")
        )
        train_losses.append(train_eval_loss)
        val_losses.append(val_loss)
        append_metrics({
            "epoch": epoch,
            "global_step": global_step,
            "tokens_seen": tokens_seen,
            "train_loss": train_eval_loss,
            "val_loss": val_loss,
        })
        if val_loader is not None and val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(BEST_CHECKPOINT_PATH, epoch, val_loss)
        save_checkpoint(OUTPUT_CHECKPOINT_PATH, epoch, val_loss)
        print(f"epoch={epoch} completed train={train_eval_loss:.4f} val={val_loss:.4f}")

    distributed_barrier()

if runtime.is_main_process:
    print(f"Saved resumed checkpoint: {OUTPUT_CHECKPOINT_PATH}")
else:
    print(f"Rank {runtime.rank}/{runtime.world_size} finished resume sync at global_step={global_step}")

In [ ]:
sample = (
    generate_protein_text(
        model,
        tokenizer,
        prompt="<|protein|>",
        device=device,
        max_new_tokens=80,
        context_length=context_length,
    )
    if runtime.is_main_process
    else f"Sample generation is emitted on rank 0 only. This rank is {runtime.rank}."
)
sample